# Notebook 03 — Long-Context Multi-Question Retrieval with RLM

This notebook demonstrates the core strength of Recursive Language Models:
**retrieving information buried inside a long document** that would overwhelm
a single LLM context window.

We build a synthetic "corporate report" (~15 pages of text) with **five facts
hidden at random positions**, then ask the RLM to find **all five** in a
single call.  The agent must decompose the document, search each chunk, and
aggregate the results — exactly the workflow RLMs are designed for.

### Why this showcases RLM's power

| Traditional LLM | RLM |
|---|---|
| Must fit the entire document in the prompt | Document stays as a Python variable (`rlm_context`) |
| Struggles when the answer is far from the end | Agent can binary-search or scan programmatically |
| One-shot retrieval only | Recursive decomposition — search, re-search, aggregate |
| No visibility into the process | Full call tree + agent steps + LLM request traces |

### What you will see

1. **Synthetic document generation** — random filler with five hidden facts.
2. **Single RLM call** — the agent decides how to split and search.
3. **Call-tree inspection** — view the recursive decomposition structure.
4. **HTML visualization** — save a fully interactive trace viewer.
5. **Accuracy check** — verify all five facts were found.


## Setup

In [16]:
import os
import sys
import json
import random
import pathlib
import importlib
from openai import OpenAI

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

LLAMA_BASE_URL = os.environ.get('LLAMA_BASE_URL', 'http://host-gateway:1337/v1')
LLAMA_MODEL    = os.environ.get('LLAMA_MODEL',    'local-model')
OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'not-needed')

import rlm_smolagent as rlm_smolagent_module
rlm_smolagent_module = importlib.reload(rlm_smolagent_module)
RLMAgent = rlm_smolagent_module.RLMAgent

from rlm_visualizer import save_html, save_json, load_json

log_dir = pathlib.Path.cwd().resolve() / 'logs'
log_dir.mkdir(parents=True, exist_ok=True)


def make_agent(max_depth=3, max_steps=12, verbose=True, capture_prompt_traces=True):
    return RLMAgent(
        base_url=LLAMA_BASE_URL,
        model_name=LLAMA_MODEL,
        api_key=OPENAI_API_KEY,
        max_depth=max_depth,
        max_steps=max_steps,
        verbose=verbose,
        capture_prompt_traces=capture_prompt_traces,
    )

print('Setup complete.')

Setup complete.


---
## Step 1 — Build a synthetic corporate report

We generate a ~6,000-word document with random business-sounding paragraphs.
Five facts are planted at random positions.  Each fact follows the pattern
`[FACT_N]: <value>` so we can programmatically verify the RLM's answers later.

The RLM will receive this as `rlm_context` — it never appears in the prompt.

You can also set `FILLER_SOURCE=local-llm` to ask the locally hosted OpenAI-compatible
server to generate extra filler paragraphs, or `FILLER_SOURCE=hybrid` to keep the
static template and append LLM-generated filler on top of it.


In [3]:
random.seed(42)

# ── Five hidden facts ──
FACTS = {
    "FACT_1": f"Q3 revenue was ${random.randint(50, 999)}.{random.randint(10,99)}M",
    "FACT_2": f"The new CEO is {random.choice(['Ryland Grace', 'Mark Watney', 'Jasmine Bashara', 'Ellen Ripley'])}",
    "FACT_3": f"Employee headcount reached {random.randint(8000, 25000):,}",
    "FACT_4": f"The product launch date is {random.choice(['March', 'June', 'September', 'November'])} {random.randint(1,28)}, {random.choice([2025, 2026])}",
    "FACT_5": f"Server uptime was {random.uniform(99.5, 99.999):.3f}%",
}

print("Hidden facts (ground truth):")
for k, v in FACTS.items():
    print(f"  {k}: {v}")

Hidden facts (ground truth):
  FACT_1: Q3 revenue was $704.24M
  FACT_2: The new CEO is Ryland Grace
  FACT_3: Employee headcount reached 17,012
  FACT_4: The product launch date is June 8, 2025
  FACT_5: Server uptime was 99.867%


In [6]:
# ── Build the filler document ──
FILLER_SOURCE = 'hybrid'  # Options: 'template', 'local-llm', 'hybrid'
EXTRA_LLM_FILLER_PARAGRAPHS = int(8)

FILLER_PARAGRAPHS = [
    "Global operations continued to expand across all geographic segments. "
    "The leadership team reviewed quarterly milestones and reaffirmed commitments "
    "to sustainability targets outlined in the annual report.",

    "Cross-functional teams delivered incremental improvements to the core platform. "
    "Internal tooling adoption increased by double digits, reducing average ticket "
    "resolution time and improving developer velocity metrics.",

    "Market conditions remained mixed, with emerging-market currencies experiencing "
    "volatility while developed-market indices posted moderate gains. The treasury "
    "team maintained a hedging strategy aligned with board-approved risk parameters.",

    "Customer satisfaction scores trended upward following the release of the "
    "redesigned mobile experience. Net Promoter Score reached an all-time high "
    "for the second consecutive quarter.",

    "Supply chain resilience initiatives showed measurable results. Dual-sourcing "
    "programs covered 92% of critical components, up from 78% in the prior year. "
    "Lead-time variability decreased substantially.",

    "Research and development expenditure increased in line with the five-year "
    "innovation roadmap. Patent filings rose across the biotechnology and "
    "advanced-materials divisions.",

    "Regulatory compliance audits concluded without material findings. The legal "
    "team continued to monitor evolving data-privacy legislation in key jurisdictions.",

    "Talent acquisition pipelines remained robust despite a competitive labour market. "
    "Early-career programs attracted a record number of applicants, with acceptance "
    "rates holding steady year over year.",

    "The board of directors approved a share-buyback programme and maintained the "
    "quarterly dividend at the previously announced level. Capital allocation "
    "priorities were reviewed during the annual strategy offsite.",

    "Cloud migration workloads progressed on schedule. The infrastructure team "
    "deprecated two legacy data centres and consolidated workloads into three "
    "primary availability zones.",
]


def _coerce_paragraphs(raw_content: str, expected_count: int) -> list[str]:
    try:
        parsed = json.loads(raw_content)
    except json.JSONDecodeError:
        parsed = None
        start = raw_content.find('[')
        end = raw_content.rfind(']')
        if start != -1 and end != -1 and end > start:
            try:
                parsed = json.loads(raw_content[start:end + 1])
            except json.JSONDecodeError:
                parsed = None

    if not isinstance(parsed, list):
        parsed = []

    cleaned = [str(item).strip() for item in parsed if str(item).strip()]
    if not cleaned:
        cleaned = [line.strip("-• \t") for line in raw_content.splitlines() if line.strip()]
    return cleaned[:expected_count]


def generate_local_filler_paragraphs(count: int) -> list[str]:
    if count <= 0:
        return []

    client = OpenAI(base_url=LLAMA_BASE_URL, api_key=OPENAI_API_KEY)
    response = client.chat.completions.create(
        model=LLAMA_MODEL,
        temperature=0.7,
        max_tokens=max(600, count * 120),
        messages=[
            {
                'role': 'system',
                'content': (
                    'You write realistic but generic corporate-report filler. '
                    'Avoid factual claims, avoid repeating phrases, and keep the tone polished.'
                ),
            },
            {
                'role': 'user',
                'content': (
                    f'Generate {count} distinct filler paragraphs for a synthetic corporate report. '
                    'Each paragraph should be 2 to 3 sentences, self-contained, and unrelated to '
                    'the hidden facts. Return only a valid JSON array of strings. Do not use markdown '
                    'or code fences.'
                ),
            },
        ],
    )
    raw_content = response.choices[0].message.content or '[]'
    return _coerce_paragraphs(raw_content, count)


# Build ~120 paragraphs of filler
base_paragraphs = [FILLER_PARAGRAPHS[i % len(FILLER_PARAGRAPHS)] for i in range(120)]
doc_paragraphs = list(base_paragraphs)
generated_paragraphs = []

if FILLER_SOURCE in {'local-llm', 'hybrid'}:
    try:
        generated_paragraphs = generate_local_filler_paragraphs(EXTRA_LLM_FILLER_PARAGRAPHS)
        if generated_paragraphs:
            doc_paragraphs.extend(generated_paragraphs)
        else:
            print('Local LLM filler generation returned no usable paragraphs; using template filler only.')
    except Exception as exc:
        print(f'Local LLM filler generation failed: {exc}')
        print('Falling back to template filler only.')
elif FILLER_SOURCE != 'template':
    print(f"Unknown FILLER_SOURCE={FILLER_SOURCE!r}; using template filler only.")

# Insert the five facts at random positions throughout the document
fact_positions = sorted(random.sample(range(10, len(doc_paragraphs) - 10), 5))
for pos, (key, value) in zip(fact_positions, FACTS.items()):
    doc_paragraphs.insert(pos, f"[{key}]: {value}")

REPORT = "\n\n".join(doc_paragraphs)

print(f'Filler source   : {FILLER_SOURCE}')
print(f'Template paras  : {len(base_paragraphs)}')
print(f'LLM paras       : {len(generated_paragraphs)}')
print(f'Document length : {len(REPORT):,} characters')
print(f'Paragraph count : {len(doc_paragraphs)}')
print(f'Approx. words   : {len(REPORT.split()):,}')
print(f'\nFact positions (paragraph indices): {fact_positions}')
print(f'\nFirst 500 characters:\n{REPORT[:500]}...')

Filler source   : hybrid
Template paras  : 120
LLM paras       : 1
Document length : 23,789 characters
Paragraph count : 126
Approx. words   : 2,972

Fact positions (paragraph indices): [13, 39, 74, 81, 87]

First 500 characters:
Global operations continued to expand across all geographic segments. The leadership team reviewed quarterly milestones and reaffirmed commitments to sustainability targets outlined in the annual report.

Cross-functional teams delivered incremental improvements to the core platform. Internal tooling adoption increased by double digits, reducing average ticket resolution time and improving developer velocity metrics.

Market conditions remained mixed, with emerging-market currencies experiencing...


---
## Step 2 — Run the RLM agent

We give the agent a single task: **find all five `[FACT_N]` entries and return
their values**.

The agent receives the full document as `context` (stored as `rlm_context` in
the REPL).  It must decide on its own how to decompose and search the text.

Typical strategies the agent may choose:
- **Binary split** → `rlm_call` on each half, recursively narrowing down.
- **Regex scan** → `import re; re.findall(...)` directly in Python.
- **Chunk-and-search** → split into N chunks, `llm_call` each one.


In [7]:
agent = make_agent(max_depth=3, max_steps=15, verbose=True)

result = agent.completion(
    task=(
        "`rlm_context` is a long corporate report (~6,000 words). "
        "Hidden throughout the text are exactly five labelled facts, each on its own line "
        "in the format `[FACT_N]: <value>` where N is 1 through 5.\n\n"
        "Your job:\n"
        "1. Search `rlm_context` to find ALL five facts.\n"
        "2. Return a numbered list with the exact fact labels and values.\n\n"
        "Strategy hints:\n"
        "- You can use Python (regex, string search, slicing) to locate the facts.\n"
        "- For large contexts, split into halves with rlm_call or scan with code.\n"
        "- Do NOT embed the full context in any sub-call — always slice first.\n"
    ),
    context=REPORT,
)

print('\n=== RLM Answer ===')
print(result.response)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ You are an RLM (Recursive Language Model) agent at recursion depth 0/3.                                         │
│                                                                                                                 │
│ You run inside a Python REPL.  The input context is available as the                                            │
│ Python variable `rlm_context` — treat it as a Python object.  Slice it,                                         │
│ search it, split it, transform it.  Do NOT embed its raw content as a                                           │
│ string literal inside any sub-call argument.                                                                    │
│                                                                                                                 │
│ Two tools are available for making LLM sub-calls:                                                               │
│                                                                                                                 │
│ `llm_call(sub_task, context_slice)`:                                                                            │
│     Direct, non-recursive LLM call.  Fast and lightweight.                                                      │
│     Use for leaf-level queries on chunks that are small enough to                                               │
│     answer in a single LLM call without further decomposition.                                                  │
│                                                                                                                 │
│ `rlm_call(sub_task, context_slice)`:                                                                            │
│     Recursive RLM sub-call.  The child agent gets its own Python REPL                                           │
│     and can decompose the slice further.  Use for complex sub-tasks                                             │
│     that may themselves need recursive processing.                                                              │
│                                                                                                                 │
│ You decide HOW to orchestrate — use any Python logic to split, filter,                                          │
│ or transform `rlm_context` before passing slices to sub-calls.                                                  │
│                                                                                                                 │
│ Example — summarise paragraph-by-paragraph with direct LLM calls:                                               │
│     paragraphs = [p for p in rlm_context.split("\n\n") if p.strip()\]                                           │
│     summaries = [llm_call(f"Summarise paragraph {i+1}", p)                                                      │
│                  for i, p in enumerate(paragraphs)\]                                                            │
│     final_answer("\n".join(summaries))                                                                          │
│                                                                                                                 │
│ Example — recursive binary split for very large contexts:                                                       │
│     mid   = len(rlm_context) // 2                                                                               │
│     left  = rlm_call("Analyse first half",  rlm_context[:mid\])                                                 │
│     right = rlm_call("Analyse second half", rlm_context[mid:\])                                                 │
│     final_answer(left + " " + right)                  

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  import re                                                                                                        
                                                                                                                   
  # Search for all FACT patterns in the context                                                                    
  fact_pattern = r'\[FACT_\d+\]:\s*(.+)'                                                                           
  matches = re.findall(fact_pattern, rlm_context)                                                                  
  print(f"Found {len(matches)} facts:")                                                                            
  for i, match in enumerate(matches, 1):                                                                           
      print(f"Fact {i}: {match}")                                                                                  
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Found 5 facts:
Fact 1: Q3 revenue was $704.24M
Fact 2: The new CEO is Ryland Grace
Fact 3: Employee headcount reached 17,012
Fact 4: The product launch date is June 8, 2025
Fact 5: Server uptime was 99.867%

Out: None

[Step 1: Duration 3.78 seconds| Input tokens: 3,252 | Output tokens: 175]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  # Extract the fact labels and values with full labels                                                            
  import re                                                                                                        
                                                                                                                   
  fact_pattern = r'\[FACT_(\d+)\]:\s*(.+)'                                                                         
  matches = re.findall(fact_pattern, rlm_context)                                                                  
                                                                                                                   
  # Create numbered list with exact labels                                                                         
  fact_list = "\n".join([f"[FACT_{i}]: {value}" for i, value in matches])                                          
  final_answer(fact_list)                                                                                          
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: [FACT_1]: Q3 revenue was $704.24M
[FACT_2]: The new CEO is Ryland Grace
[FACT_3]: Employee headcount reached 17,012
[FACT_4]: The product launch date is June 8, 2025
[FACT_5]: Server uptime was 99.867%

[Step 2: Duration 3.87 seconds| Input tokens: 6,820 | Output tokens: 340]


=== RLM Answer ===
[FACT_1]: Q3 revenue was $704.24M
[FACT_2]: The new CEO is Ryland Grace
[FACT_3]: Employee headcount reached 17,012
[FACT_4]: The product launch date is June 8, 2025
[FACT_5]: Server uptime was 99.867%


---
## Step 3 — Inspect the call tree

The metadata shows exactly how the agent decomposed the task: which child
calls were made, at what depth, and how large each context slice was.


In [ ]:
def print_tree(node: dict, indent: int = 0) -> None:
    prefix = '  ' * indent
    depth  = node.get('depth', '?')
    dur    = node.get('duration_s', '?')
    task   = node.get('task_preview', '')
    resp   = node.get('response_preview', '')
    ctx    = node.get('context_size', 0)
    steps  = len(node.get('agent_steps', []))
    reqs   = len(node.get('llm_requests', []))
    kids   = len(node.get('children', []))
    print(f"{prefix}[depth={depth}] ({dur}s) ctx={ctx:,}B  steps={steps}  llm_reqs={reqs}  children={kids}")
    print(f"{prefix}  task    : {task}")
    print(f"{prefix}  response: {resp}")
    for child in node.get('children', []):
        print_tree(child, indent + 1)

print('=== Call Tree ===')
print_tree(result.metadata['call_tree'])

---
## Step 4 — View the agent's Python code steps

Each node in the tree stores the code actions the agent executed in its REPL.
This is where you can see exactly *how* the agent chose to decompose the task.


In [ ]:
def print_agent_steps(node: dict, indent: int = 0) -> None:
    prefix = '  ' * indent
    depth = node.get('depth', '?')
    task = node.get('task_preview', '')
    steps = node.get('agent_steps', [])

    if steps:
        print(f"{prefix}── Node depth={depth}: {task}")
        for s in steps:
            num = s.get('step_number', '?')
            code_action = s.get('code_action', '')
            obs = s.get('observations', '')
            is_final = s.get('is_final_answer', False)
            print(f"{prefix}  Step {num} {'[FINAL]' if is_final else ''}:")
            if code_action:
                for line in code_action.split('\n'):
                    print(f"{prefix}    >>> {line}")
            if obs:
                for line in str(obs).split('\n')[:5]:
                    print(f"{prefix}    ... {line}")
        print()

    for child in node.get('children', []):
        print_agent_steps(child, indent + 1)

print('=== Agent Steps ===')
print_agent_steps(result.metadata['call_tree'])

---
## Step 5 — Verify accuracy

We check whether the RLM's answer contains all five ground-truth facts.


In [8]:
answer = result.response
print("=== Accuracy Check ===")
found = 0
for key, value in FACTS.items():
    # Check if the core value (without the label prefix) is in the answer
    if value in answer or key in answer:
        print(f"  ✅ {key}: {value}")
        found += 1
    else:
        print(f"  ❌ {key}: {value}  — NOT FOUND in answer")

print(f"\nScore: {found}/5 facts retrieved")
if found == 5:
    print("🎉 Perfect retrieval — all facts found!")
elif found >= 3:
    print("👍 Good retrieval — most facts found.")
else:
    print("⚠️  The model may need more steps or a different strategy.")

=== Accuracy Check ===
  ✅ FACT_1: Q3 revenue was $704.24M
  ✅ FACT_2: The new CEO is Ryland Grace
  ✅ FACT_3: Employee headcount reached 17,012
  ✅ FACT_4: The product launch date is June 8, 2025
  ✅ FACT_5: Server uptime was 99.867%

Score: 5/5 facts retrieved
🎉 Perfect retrieval — all facts found!


---
## Step 6 — Save the interactive HTML visualization

The HTML visualizer provides a full interactive view of:
- The recursive call tree (left panel)
- Node details, agent steps, and LLM request payloads (right panel)

Open the generated file in any browser — no server required.


In [17]:
log_dir.mkdir(parents=True, exist_ok=True)

html_path = save_html(result, log_dir / 'long_context_qa.html')
print(f'HTML visualization saved to: {html_path}')

# Also save the raw JSON for later re-visualization
json_path = save_json(result, log_dir / 'long_context_qa.json')
print(f'JSON trace saved to: {json_path}')

# You can also use the convenience methods:
# result.save_html('trace.html')
# result.save_json('trace.json')

HTML visualization saved to: /workspace/notebooks/logs/long_context_qa.html
JSON trace saved to: /workspace/notebooks/logs/long_context_qa.json


---
## Step 7 — Reload and re-visualize from JSON

You can reload a saved JSON trace and regenerate the HTML visualization
without re-running the agent.  This is useful for sharing results or
creating visualizations from archived runs.


In [ ]:
# Reload from JSON
reloaded = load_json(log_dir / 'long_context_qa.json')
print(f"Reloaded response preview: {reloaded['response'][:200]}...")

# Regenerate HTML from the reloaded data
html_path_2 = save_html(reloaded, log_dir / 'long_context_qa_reloaded.html')
print(f'Reloaded HTML saved to: {html_path_2}')

---
## What to try next

- **Increase difficulty**: bump the paragraph count to 500+ or reduce `max_steps`.
- **Change the strategy hint**: remove the regex hint and see if the agent
  still finds all facts via recursive splitting.
- **Lower `max_depth`**: force the agent to handle larger chunks per node.
- **Compare with a plain LLM call**: remove the `context` parameter and embed
  the full document in the prompt — watch it fail on long inputs.
- **Try different tasks**: instead of factual retrieval, ask for a summary,
  sentiment analysis, or question-answering across the document.

The HTML visualizer makes it easy to compare different runs side by side.
